# Imports & Loads

In [ ]:
from pathlib import Path

DATA_DIR = Path("../data")

TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (balanced_accuracy_score, confusion_matrix,
                             classification_report)
from lightgbm import LGBMClassifier

import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))
from utils.evaluation import evaluate_oof
from utils.feature_engineering import engineer_features
from utils.probes import train_probe
from utils.final_probe import train_final_probe
from utils.predict_proba import predict_probe
import importlib
import utils.feature_engineering
from utils.feature_engineering import prepare_fold
from utils.error_analysis import analyze_errors

import joblib
import warnings
warnings.filterwarnings('ignore')

N_FOLDS   = 5
SEED      = 42
N_CLASSES = 3

train    = pd.read_parquet(TRAIN_PATH)
test     = pd.read_parquet(TEST_PATH)

TARGET       = 'health_condition'
CLASS_COLORS = {'fit': '#55A868', 'at-risk': '#4C72B0', 'unhealthy': '#C44E52'}

CAT_COLS = ['diet_type', 'stress_level', 'sleep_quality',
            'physical_activity_level', 'smoking_alcohol', 'gender']
NUM_COLS = ['sleep_duration', 'heart_rate', 'bmi', 'calorie_expenditure',
            'step_count', 'exercise_duration', 'water_intake']

for df in [train, test]:
    for c in CAT_COLS:
        df[c] = df[c].astype(str).str.strip().str.lower().replace('nan', np.nan)

print(f'Train    : {train.shape}')
print(f'Test     : {test.shape}')

Train    : (690088, 14)
Test     : (295753, 13)


In [5]:
STRESS_MISSING_TR = train['stress_level'].isna().values
STRESS_MISSING_TE = test['stress_level'].isna().values
SLEEP_MISSING_TR  = train['sleep_duration'].isna().values
SLEEP_MISSING_TE  = test['sleep_duration'].isna().values

print(f'train stress missing: {STRESS_MISSING_TR.sum():,} ({STRESS_MISSING_TR.mean()*100:.1f}%)')
print(f'test  stress missing: {STRESS_MISSING_TE.sum():,} ({STRESS_MISSING_TE.mean()*100:.1f}%)')
print(f'train sleep_duration missing : {SLEEP_MISSING_TR.sum():}({SLEEP_MISSING_TR.mean()*100:.1f}%')
print(f'test Sleep_duration missng   : {SLEEP_MISSING_TE.sum():}({SLEEP_MISSING_TE.mean()*100:.1f}%)')

train stress missing: 0 (0.0%)
test  stress missing: 0 (0.0%)
train sleep_duration missing : 75999(11.0%
test Sleep_duration missng   : 32571(11.0%)


# Feature Engineering

In [3]:
train    = engineer_features(train)
test     = engineer_features(test)

ENGINEERED_CATS = ['stress_x_activity', 'stress_x_sleepq', 'stress_x_smoke','sleep_x_stress']
ENGINEERED_NUMS = ['sleep_lt6', 'sleep_ge7', 'steps_x_exercise',
                   'activity_efficiency', 'sleep_x_steps', 'bmi_band', 'sleep_dev']

FEATURES  = NUM_COLS + CAT_COLS + ENGINEERED_CATS + ENGINEERED_NUMS
MODEL_CATS = CAT_COLS + ENGINEERED_CATS
print(f'{len(FEATURES)} features ({len(MODEL_CATS)} categorical)')

24 features (10 categorical)


# Stress Level Recovery Probe

In [5]:
final_stress_probe = joblib.load('../models/stress_probe_1.pkl')

probs_train_stress = predict_probe(train,final_stress_probe)

le = final_stress_probe['label_encoder']

for i, cls in enumerate(le.classes_):
    train[f"p_stress_{cls}"] = probs_train_stress[:, i]

probs_test_stress = predict_probe(test,final_stress_probe)
for i,cls in enumerate(le.classes_):
    test[f'p_stress_{cls}'] = probs_test_stress[:,i]

In [6]:
# Updating FEATURES and MODEL_CATS varable beacuse 3 new features just got added

FEATURES = [col for col in train.columns.tolist() if col !=TARGET]
MODEL_CATS = [col for col in train.select_dtypes(include='object').columns.tolist() if col != TARGET]

# Sleep Duration Recovery Probe

In [7]:
final_sleep_probe = joblib.load("../models/sleep_probe_1.pkl")

probs_train_sleep = predict_probe(train,final_sleep_probe)

le = final_sleep_probe['label_encoder']

for i, cls in enumerate(le.classes_):
    train[f"p_sleep_{cls}"] = probs_train_sleep[:, i]

probs_test_sleep = predict_probe(test,final_sleep_probe)

for i,cls in enumerate(le.classes_):
    test[f'p_sleep_{cls}'] = probs_test_sleep[:,i]

In [8]:
# Updating FEATURES and MODEL_CATS varable beacuse 3 new features just got added

FEATURES = [col for col in train.columns.tolist() if col !=TARGET]
MODEL_CATS = [col for col in train.select_dtypes(include='object').columns.tolist() if col != TARGET]

In [9]:
FEATURES

['sleep_duration',
 'heart_rate',
 'bmi',
 'calorie_expenditure',
 'step_count',
 'exercise_duration',
 'water_intake',
 'diet_type',
 'stress_level',
 'sleep_quality',
 'physical_activity_level',
 'smoking_alcohol',
 'gender',
 'stress_x_activity',
 'stress_x_sleepq',
 'stress_x_smoke',
 'sleep_x_stress',
 'sleep_lt6',
 'sleep_ge7',
 'steps_x_exercise',
 'activity_efficiency',
 'sleep_x_steps',
 'bmi_band',
 'sleep_dev',
 'p_stress_high',
 'p_stress_low',
 'p_stress_medium',
 'p_sleep_0',
 'p_sleep_1',
 'p_sleep_2']

# Training single Catboost model on entire dataset

In [ ]:
le = LabelEncoder()
y = le.fit_transform(train[TARGET])
print("Class encoding: ",dict(zip(le.classes_,range(len(le.classes_)))))

models = []
cat_cols = [col for col in train.select_dtypes(include = 'object').columns if col != TARGET]

Class encoding:  {'at-risk': 0, 'fit': 1, 'unhealthy': 2}


CatBoost requires categorical featues as strings with no NAN - fill missing with an explicit **missing** token. Numeric NaNs are handled natively.

In [12]:
for df in [train, test]:
    for c in MODEL_CATS:
        df[c] = df[c].fillna('missing').astype(str)

print('Cat features ready:', MODEL_CATS)
print(train[MODEL_CATS].isna().sum().sum(), 'remaining NaN in cats (should be 0)')

Cat features ready: ['diet_type', 'stress_level', 'sleep_quality', 'physical_activity_level', 'smoking_alcohol', 'gender', 'stress_x_activity', 'stress_x_sleepq', 'stress_x_smoke', 'sleep_x_stress']
0 remaining NaN in cats (should be 0)


In [13]:
CAT_PARAMS = dict(
    loss_function      = 'MultiClass',
    eval_metric        = 'MultiClass',
    iterations         = 898,
    learning_rate      = 0.049761801548562384,
    depth              = 5,
    l2_leaf_reg        = 8.432180184293285,
    random_strength    = 0.1614256313595708,
    bagging_temperature= 0.2727186451306435,
    min_data_in_leaf   = 24,
    auto_class_weights = 'Balanced',
    random_seed        = SEED,
    verbose            = 0,
    allow_writing_files= False,
    task_type          = 'GPU',
    devices            = '0',
    border_count       = 254,
    one_hot_max_size   = 16, 
)

In [19]:
from utils.feature_engineering import fit_target_encoding, transform_target_encoding

encoding_info = fit_target_encoding(train,MODEL_CATS,target= TARGET)
train_fold = transform_target_encoding(train,encoding_info)

FEATURES = [
    col for col in train_fold.columns
    if col != TARGET
]

CAT_FEATURES = [
    col for col in train_fold[FEATURES].select_dtypes(include="object").columns
]

CAT_IDX = [
    FEATURES.index(col)
    for col in CAT_FEATURES
]

train_pool = Pool(
    train_fold[FEATURES],
    y,
    cat_features=CAT_IDX
)

model = CatBoostClassifier(**CAT_PARAMS)

model.fit(train_pool)


In [ ]:
artifacts = {
    "model": model,
    "label_encoder": le,

    "features": FEATURES,
    "categorical_features": CAT_FEATURES,

    "target_encoding_info": encoding_info,

    "sleep_probe": final_sleep_probe,
    "stress_probe": final_stress_probe,
}

joblib.dump(
    artifacts,
    "../models/final_catboost.pkl"
)

['../models/final_catboost.pkl']